# 02 — Build resized study cache

Run on **Kaggle** with competition data attached (GPU optional; CPU OK but slower).

Creates `/kaggle/working/cache_v1/{StudyUID}.npz` then Save Version → **New Dataset** from output.

Settings tip: start with `LIMIT=50` smoke test, then full run overnight.

In [ ]:
from pathlib import Path
import sys, subprocess

# If you uploaded this repo as a Kaggle dataset, point REPO to it.
REPO_CANDIDATES = [
    Path('/kaggle/input/rsna-knee-code'),
    Path('/kaggle/input/rsna-knee-abnormality-detection-model'),
    Path('/kaggle/working/repo'),
]
REPO = next((p for p in REPO_CANDIDATES if (p / 'src' / 'rsna_knee').exists()), None)
if REPO is None:
    # Clone from private repo only if internet+token available; otherwise upload code dataset.
    raise SystemExit('Attach a Kaggle Dataset of this git repo (src/rsna_knee must exist)')
sys.path.insert(0, str(REPO / 'src'))
print('REPO', REPO)

DATA_CANDIDATES = [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/rsna-knee-abnormalities-detection'),
]
DATA = next(p for p in DATA_CANDIDATES if (p / 'train.csv').exists())
print('DATA', DATA)

LIMIT = 50  # set 0 for full 4407 studies
OUT = Path('/kaggle/working/cache_v1')
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
%pip -q install pydicom opencv-python-headless tqdm pyyaml

cmd = [
    sys.executable, str(REPO / 'scripts' / 'build_cache.py'),
    '--train-csv', str(DATA / 'train.csv'),
    '--series-csv', str(DATA / 'train_series.csv'),
    '--series-root', str(DATA / 'train_series'),
    '--out-dir', str(OUT),
    '--max-series', '3',
    '--n-slices', '12',
    '--image-size', '224',
]
if LIMIT:
    cmd += ['--limit', str(LIMIT)]
print(' '.join(cmd))
subprocess.check_call(cmd)
print('cache files', len(list(OUT.glob('*.npz'))))

After a full successful run: **Save Version** → create Dataset e.g. `rsna-knee-cache-v1` from the output.
Estimated size ~few GB (uint8, 3 series × 12 slices × 224).